In [1]:
import os
import csv
from geo.Geoserver import Geoserver

In [ ]:
def find_geospatial_files(root_dir, output_csv='geospatial_files.csv'):
    """
    Searches for .tif and .shp files in all subdirectories of root_dir.
    Outputs a CSV file with columns:
    1. Main subfolder name
    2. Full file path
    """
    file_entries = []


    # os.walk walks through the directory tree rooted at root_dir.
    # It yields a 3-tuple (dirpath, dirnames, filenames):
    # - dirpath: the current path
    # - dirnames: list of subdirectories in dirpath
    # - filenames: list of files in dirpath
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for file in filenames:
            if file.lower().endswith(('.tif', '.shp')):
                full_path = os.path.join(dirpath, file)
                
                # Get the relative path from the root directory to the current directory
                rel_path = os.path.relpath(dirpath, root_dir)
                
                # Extract the first folder under the root directory (the "main subfolder")
                # If we're still in the root directory itself, just use its name
                main_subfolder = rel_path.split(os.sep)[0] if rel_path != '.' else os.path.basename(root_dir)
                
                # Add the (main_subfolder, full_path) tuple to our list
                file_entries.append((main_subfolder, full_path))

    # Write the results to a CSV file
    # newline='' avoids blank lines on Windows
    # encoding='utf-8' ensures support for special characters in paths
    with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Main Subfolder', 'Full File Path'])
        writer.writerows(file_entries)

    print(f"Found {len(file_entries)} geospatial files. Saved to '{output_csv}'.")

In [3]:
# Example usage
if __name__ == "__main__":
    root_folder = "Z:\data"  # Change this to your root folder path
    find_geospatial_files(root_folder)

Found 175267 geospatial files. Saved to 'geospatial_files.csv'.


In [2]:
def list_geoserver_layers(geoserver_url, username, password, output_csv='geoserver_layers.csv'):
    """
    Connects to a GeoServer instance and lists all layers with their corresponding store paths.
    
    Parameters:
    - geoserver_url (str): The base URL of the GeoServer REST API (e.g. http://localhost:8080/geoserver)
    - username (str): GeoServer admin username
    - password (str): GeoServer admin password
    - output_csv (str): Name of the output CSV file
    """
    # Connect to GeoServer
    gs = Geoserver(geoserver_url, username=username, password=password)

    # Fetch all layers
    all_layers = gs.get_layer()  # Returns a list of dictionaries

    # Prepare a list to hold (layer_name, workspace/store) entries
    layer_store_table = []

    for layer in all_layers:
        layer_name = layer.get('name')

        # Fetch full layer info to extract store details
        layer_details = gs.get_layer(layer_name)
        resource = layer_details.get('resource', {})

        # Extract full path: workspace:store
        store_path = resource.get('href', '')
        # Example href: http://localhost:8080/geoserver/rest/workspaces/myws/datastores/mystore/featuretypes/mylayer

        # Try to extract /workspaces/{workspace}/datastores/{store}
        store_segment = ''
        if '/workspaces/' in store_path:
            parts = store_path.split('/')
            try:
                ws_index = parts.index('workspaces')
                workspace = parts[ws_index + 1]
                if 'datastores' in parts:
                    store_index = parts.index('datastores')
                    store = parts[store_index + 1]
                    store_segment = f'{workspace}/{store}'
                elif 'coveragestores' in parts:
                    store_index = parts.index('coveragestores')
                    store = parts[store_index + 1]
                    store_segment = f'{workspace}/{store}'
            except (ValueError, IndexError):
                store_segment = 'Unknown'

        layer_store_table.append((layer_name, store_segment))

    # Write to CSV
    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Layer Name', 'Store Path (workspace/store)'])
        writer.writerows(layer_store_table)

    print(f"{len(layer_store_table)} layers exported to '{output_csv}'.")




In [ ]:
if __name__ == "__main__":
    geoserver_url = 'http://localhost:8080/geoserver'  # Replace with your GeoServer URL
    username = 'admin'
    password = 'geoserver'
    list_geoserver_layers(geoserver_url, username, password)